# Does holding the middle bus help passengers?

When Bus 1 is delayed, does briefly holding Bus 2 before Market improve passenger journeys? Three buses travel from Origin through Market and School to Terminal. We compare the same passengers and road conditions with and without holding.

## Model and assumptions

Buses dispatch at 1, 7 and 13 minutes. Passenger arrivals are Poisson, sampled with exponential inter-arrival times. New arrivals stop at the third dispatch so three buses can clear the admitted cohort. Dwell is $5.264+2.136B+1.503A$ seconds, with one FIFO boarding berth per stop. Boarding and alighting are recorded at service start. Link means are the first three Road 1 values in Herath et al. (2024), Table 4, in seconds. Lognormal road variability, demand, destinations and the disruption are scenario assumptions.

The event calendar jumps to each arrival, service start or release. The first link adds a delay to Bus 1. At Market, Bus 2 can be held for $\min(h_{\max},\max(0,360-g))$ seconds, where $g$ is the time since the previous release from the control point. Buses 1 and 3 have zero deliberate holding. A bus may still queue behind another vehicle. We measure platform wait to boarding and journey time to alighting. The School gaps use **boarding start** times.

In [ ]:
# Install the plotting dependency in a fresh Colab runtime.
from importlib.util import find_spec

if find_spec("google") is not None and find_spec("google.colab") is not None:
    from subprocess import check_call
    from sys import executable

    check_call([executable, "-m", "pip", "install", "matplotlib>=3.7"])

In [ ]:
from collections import deque
from heapq import heappop, heappush
from itertools import count
from math import isclose, isfinite
from statistics import fmean

EVENT_PRIORITY = {
    "berth_release": 0,
    "passenger_arrival": 1,
    "control_arrival": 2,
    "control_release": 3,
    "bus_arrival": 4,
    "service_start": 5,
}
KANDY_DWELL = (5.264, 2.136, 1.503)


def simulate_corridor(
    stops, running_times, dispatches, trips, *, dwell=KANDY_DWELL,
    disruption=None, control_stop=None, target_gap=80, hold_cap=0,
    running_time_draws=None, one_stop_service_times=None, hold_buses=None
):
    """Run the corridor until its event calendar empties. All times are in seconds.

    The optional one-stop service times are for bus-only queue verification.
    A run with an unfinished passenger raises an error.
    """
    # Each trip contains its arrival time, origin index and destination index.
    if len(stops) != len(running_times) + 1:
        raise ValueError("Give one running time per link between stops.")
    if any(t <= 0 for t in running_times) or tuple(sorted(dispatches)) != tuple(dispatches):
        raise ValueError("Link times must be positive and dispatches must be ordered.")
    if not dispatches or dispatches[0] < 0:
        raise ValueError("Give at least one nonnegative dispatch time.")
    if running_time_draws is not None:
        if len(running_time_draws) != len(dispatches):
            raise ValueError("Give one row of running-time draws per dispatched bus.")
        for bus_times in running_time_draws:
            if len(bus_times) != len(running_times):
                raise ValueError("Give each bus one running-time draw per link.")
            if any(not isfinite(seconds) or seconds <= 0 for seconds in bus_times):
                raise ValueError("Running-time draws must be finite and positive.")
    if one_stop_service_times is not None:
        if len(stops) != 1 or trips:
            raise ValueError("Override service times only in a bus-only, one-stop run.")
        if len(one_stop_service_times) != len(dispatches) or any(
            not isfinite(seconds) or seconds <= 0 for seconds in one_stop_service_times
        ):
            raise ValueError("Give each bus a finite positive service time.")

    fixed_dwell, seconds_per_boarding, seconds_per_alighting = dwell
    if fixed_dwell <= 0 or min(seconds_per_boarding, seconds_per_alighting) < 0:
        raise ValueError("Fixed dwell must be positive and passenger coefficients nonnegative.")
    if disruption is not None:
        delayed_bus, delayed_link, extra_time = disruption
        if not (1 <= delayed_bus <= len(dispatches)
                and 0 <= delayed_link < len(running_times) and extra_time >= 0):
            raise ValueError("Disruption must identify an existing bus, link and nonnegative delay.")

    if control_stop is not None and not 0 < control_stop < len(stops) - 1:
        raise ValueError("Put the control point before a downstream boarding stop.")
    if target_gap <= 0 or hold_cap < 0:
        raise ValueError("Target gap must be positive and the hold cap nonnegative.")
    if control_stop is None and hold_cap > 0:
        raise ValueError("Choose a control stop to enable holding.")

    calendar = []
    sequence = count()
    trace = []
    # Buses queue for a berth. Passengers wait separately on the platform.
    bus_queues = [deque() for _ in stops]
    platform_queues = [deque() for _ in stops]
    berth_bus = [None for _ in stops]
    last_link_arrival = [float("-inf") for _ in running_times]
    bus_visits = {}
    incoming_road_times = {}
    passengers = []
    onboard = {bus: [] for bus in range(1, len(dispatches) + 1)}
    control_queue = deque()
    control_times = {}
    active_control_bus = None
    last_control_release = None

    def schedule(time, event, subject, stop):
        heappush(calendar, (time, EVENT_PRIORITY[event], next(sequence), event, subject, stop))

    def release_next_bus(now):
        nonlocal active_control_bus
        if active_control_bus is not None or not control_queue:
            return
        bus = control_queue.popleft()
        active_control_bus = bus
        times = control_times[bus]
        times["control_queue_wait"] = now - times["control_arrival"]

        hold = 0
        if last_control_release is not None and (hold_buses is None or bus in hold_buses):
            observed_gap = now - last_control_release
            hold = min(hold_cap, max(0, target_gap - observed_gap))
        times["hold"] = hold
        schedule(now + hold, "control_release", bus, control_stop)

    for bus, dispatch in enumerate(dispatches, start=1):
        schedule(dispatch, "bus_arrival", bus, 0)
    for rider, (arrival, origin, destination) in enumerate(trips):
        if arrival < 0 or not 0 <= origin < destination < len(stops):
            raise ValueError("Passenger trips need a nonnegative time and a downstream destination.")
        passengers.append({
            "arrival": arrival, "origin": origin, "destination": destination,
            "board": None, "alight": None, "bus": None,
        })
        schedule(arrival, "passenger_arrival", rider, origin)

    previous_time = 0
    while calendar:
        now, _, _, event, subject, stop = heappop(calendar)
        assert now >= previous_time
        previous_time = now
        trace.append((now, event, subject, stop))
        queue = bus_queues[stop]

        if event == "passenger_arrival":
            platform_queues[stop].append(subject)

        elif event == "control_arrival":
            control_times[subject] = {"control_arrival": now}
            control_queue.append(subject)
            release_next_bus(now)

        elif event == "control_release":
            assert active_control_bus == subject
            control_times[subject]["control_release"] = now
            last_control_release = now
            active_control_bus = None
            schedule(now, "bus_arrival", subject, stop)
            release_next_bus(now)

        elif event == "bus_arrival":
            visit = {"arrival": now}
            if stop > 0:
                visit.update(incoming_road_times.pop((subject, stop)))
            if stop == control_stop:
                visit.update(control_times[subject])
            bus_visits[subject, stop] = visit
            queue.append(subject)
            if berth_bus[stop] is None and len(queue) == 1:
                schedule(now, "service_start", subject, stop)

        elif event == "service_start":
            front_bus = queue.popleft()
            assert front_bus == subject and berth_bus[stop] is None
            berth_bus[stop] = subject

            remaining = []
            alightings = 0
            for rider in onboard[subject]:
                passenger = passengers[rider]
                if passenger["destination"] == stop:
                    passenger["alight"] = now
                    alightings += 1
                else:
                    remaining.append(rider)

            boardings = len(platform_queues[stop])
            while platform_queues[stop]:
                rider = platform_queues[stop].popleft()
                passengers[rider]["board"] = now
                passengers[rider]["bus"] = subject
                remaining.append(rider)
            onboard[subject] = remaining

            duration = (
                fixed_dwell
                + seconds_per_boarding * boardings
                + seconds_per_alighting * alightings
            )
            if one_stop_service_times is not None:
                duration = one_stop_service_times[subject - 1]
            visit = bus_visits[subject, stop]
            visit.update(
                service_start=now, boardings=boardings, alightings=alightings,
                berth_queue_wait=now - visit["arrival"], dwell_seconds=duration,
            )
            schedule(now + duration, "berth_release", subject, stop)

        elif event == "berth_release":
            assert berth_bus[stop] == subject
            bus_visits[subject, stop]["departure"] = now
            berth_bus[stop] = None
            if queue:
                schedule(now, "service_start", queue[0], stop)
            if stop < len(running_times):
                travel_seconds = running_times[stop]
                if running_time_draws is not None:
                    # Bus IDs start at one; rows and link indices start at zero.
                    travel_seconds = running_time_draws[subject - 1][stop]
                disruption_seconds = 0
                if disruption is not None and subject == delayed_bus and stop == delayed_link:
                    disruption_seconds = extra_time
                free_arrival = now + travel_seconds + disruption_seconds
                # Keep buses in departure order on each link.
                next_arrival = max(free_arrival, last_link_arrival[stop] + 1)
                last_link_arrival[stop] = next_arrival
                next_stop = stop + 1
                incoming_road_times[subject, next_stop] = {
                    "sampled_road_seconds": travel_seconds,
                    "road_disruption_seconds": disruption_seconds,
                    "no_overtaking_seconds": next_arrival - free_arrival,
                }
                arrival_event = "control_arrival" if next_stop == control_stop else "bus_arrival"
                schedule(next_arrival, arrival_event, subject, next_stop)

    completed = sum(rider["alight"] is not None for rider in passengers)
    still_waiting = sum(map(len, platform_queues))
    assert completed + still_waiting + sum(map(len, onboard.values())) == len(passengers)
    assert not any(onboard.values()), "All boarded passengers must alight by the terminal."
    assert len(bus_visits) == len(stops) * len(dispatches)
    assert not control_queue and active_control_bus is None
    assert not incoming_road_times
    if still_waiting:
        raise ValueError(
            f"Only {completed} of {len(passengers)} passengers completed; "
            f"{still_waiting} still waiting when service ended. Extend the dispatch schedule."
        )
    return bus_visits, passengers, trace


def passenger_time_breakdown(visits, rider):
    """Attribute one completed journey from platform arrival to start of alighting."""
    bus = rider["bus"]
    origin, destination = rider["origin"], rider["destination"]
    if rider["board"] is None or rider["alight"] is None:
        raise ValueError("A delay breakdown requires a completed journey.")

    seconds = {
        "platform_wait": rider["board"] - rider["arrival"],
        "sampled_road_seconds": 0.0,
        "road_disruption_seconds": 0.0,
        "no_overtaking_seconds": 0.0,
        "berth_queue_wait": 0.0,
        "dwell_seconds": 0.0,
        "control_queue_wait": 0.0,
        "hold": 0.0,
    }
    for stop in range(origin, destination + 1):
        visit = visits[bus, stop]
        if stop < destination:
            seconds["dwell_seconds"] += visit["dwell_seconds"]
        if stop > origin:
            for component in (
                "sampled_road_seconds", "road_disruption_seconds", "no_overtaking_seconds",
                "berth_queue_wait", "control_queue_wait", "hold",
            ):
                seconds[component] += visit.get(component, 0.0)
    return seconds

In [ ]:
from math import isfinite, log1p
from random import Random


def poisson_arrivals(rate_per_minute, end_seconds, rng):
    """Return one stop's arrival times in seconds before the cutoff."""
    if not isfinite(rate_per_minute) or rate_per_minute < 0:
        raise ValueError("The arrival rate must be finite and nonnegative.")
    if not isfinite(end_seconds) or end_seconds < 0:
        raise ValueError("The arrival cutoff must be finite and nonnegative.")
    if rate_per_minute == 0:
        return []

    mean_gap = 60 / rate_per_minute
    arrival_time = 0.0
    arrivals = []
    while arrival_time < end_seconds:
        draw = rng.random()
        if draw == 0:
            continue  # Use the open interval so every gap is positive.
        arrival_time += -mean_gap * log1p(-draw)
        if arrival_time < end_seconds:
            arrivals.append(arrival_time)
    return arrivals


def generate_passenger_trips(rates_per_minute, end_seconds, seed):
    """Return (arrival, origin, destination) for the corridor."""
    rng = Random(seed)
    terminal = len(rates_per_minute)
    trips = []
    for origin, rate in enumerate(rates_per_minute):
        for rider, arrival in enumerate(poisson_arrivals(rate, end_seconds, rng)):
            # Match the fixed-input case: every third rider makes a one-stop trip.
            destination = origin + 1 if rider % 3 == 0 else terminal
            trips.append((arrival, origin, destination))
    return sorted(trips, key=lambda trip: trip[0])

In [ ]:
from math import log, sqrt
from statistics import pstdev


def sample_link_running_times(mean_seconds, coefficient_of_variation, bus_count, seed):
    """Return a tuple of free-running times by bus and link."""
    if not mean_seconds or any(not isfinite(mean) or mean <= 0 for mean in mean_seconds):
        raise ValueError("Give each link a finite positive mean running time.")
    if not isfinite(coefficient_of_variation) or coefficient_of_variation < 0:
        raise ValueError("The coefficient of variation must be finite and nonnegative.")
    if bus_count < 1:
        raise ValueError("Sample running times for at least one bus.")

    if coefficient_of_variation == 0:
        return (tuple(mean_seconds),) * bus_count

    rng = Random(seed)
    log_variance = log1p(coefficient_of_variation ** 2)
    log_sd = sqrt(log_variance)
    log_means = [log(mean) - log_variance / 2 for mean in mean_seconds]
    return tuple(
        tuple(rng.lognormvariate(log_mean, log_sd) for log_mean in log_means)
        for _ in range(bus_count)
    )

## Verification

Check event ordering, FIFO queues, the hold cap and rejection of incomplete cohorts. Compare the same engine with the analytical M/M/1 and fixed-headway benchmarks. The last graph checks Monte Carlo error scaling against the 40-second fixed-headway result.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
Path('figures').mkdir(exist_ok=True)

In [ ]:
# One berth, fixed 30 s dwell. Bus 3 arrives at exactly the first berth release.
visits, _, trace = simulate_corridor(
    ("Test stop",), (), (70, 80, 100), (), dwell=(30, 0, 0)
)
assert visits[1, 0]["departure"] == 100
assert visits[2, 0]["service_start"] - visits[2, 0]["arrival"] == 20
assert visits[3, 0]["service_start"] == 130
assert [event for time, event, _, _ in trace if time == 100] == [
    "berth_release", "bus_arrival", "service_start",
]
print("PASS: 20 s berth wait and FIFO at a simultaneous release and arrival.")

# Hand trace with d = 10 + 2B + A, two road links of 20 and 30 seconds.
# Passenger 1 boards at 0 s; Passenger 2 arrives at 30 s and boards at 32 s.
# Both reach their destination at 74 s. The second bus queues at both downstream stops.
visits, riders, _ = simulate_corridor(
    ("Origin", "Middle", "Terminal"), (20, 30), (0, 5),
    ((0, 0, 2), (30, 1, 2)), dwell=(10, 2, 1)
)
assert visits[1, 1]["arrival"] == 32
assert visits[2, 1]["service_start"] - visits[2, 1]["arrival"] == 2
assert visits[2, 2]["service_start"] - visits[2, 2]["arrival"] == 2
assert [(rider["board"], rider["alight"]) for rider in riders] == [(0, 74), (32, 74)]
assert all(rider["bus"] == 1 for rider in riders)
print("PASS: full trips take 74 s and 44 s, with 0 s and 2 s platform waits.")

In [ ]:
# Approach arrivals are 11, 51, 56 and 151 s. The target gap is 60 s and
# the cap is 15 s. Bus 3 reaches the front only after Bus 2 releases at 66 s.
control_visits, _, _ = simulate_corridor(
    ("Origin", "Market", "Terminal"), (10, 10), (0, 40, 45, 140), (),
    dwell=(1, 0, 0), control_stop=1, target_gap=60, hold_cap=15,
)
expected_control = (
    (11, 11, 0, 0), (51, 66, 0, 15), (56, 81, 10, 15), (151, 151, 0, 0),
)
timing_fields = ("control_arrival", "control_release", "control_queue_wait", "hold")
for bus, expected in enumerate(expected_control, start=1):
    visit = control_visits[bus, 1]
    assert tuple(visit[field] for field in timing_fields) == expected
print("PASS: first release, cap, FIFO control queue and long-gap release.")

In [ ]:
# A passenger arriving after the last boarding opportunity must cause an error.
try:
    simulate_corridor(("Origin", "Terminal"), (10,), (0,), ((1, 0, 1),), dwell=(1, 0, 0))
except ValueError as error:
    assert "1 still waiting" in str(error)
else:
    raise AssertionError("An unfinished passenger was silently accepted.")
print("PASS: a short dispatch schedule is rejected with the unfinished passenger count.")

In [ ]:
from itertools import accumulate

def verify_mm1(replications=8, warmup_buses=1_000, measured_buses=4_000):
    # Rates in buses per second. The one-stop engine handles all FIFO events.
    arrival_rate, service_rate = 0.5 / 60, 1 / 60
    total_buses = warmup_buses + measured_buses
    expected = arrival_rate / (service_rate * (service_rate - arrival_rate))
    estimates = []
    for rep in range(replications):
        arrival_rng = Random(1100 + rep)
        service_rng = Random(2100 + rep)
        arrivals = tuple(accumulate(arrival_rng.expovariate(arrival_rate)
                                    for _ in range(total_buses)))
        service = tuple(service_rng.expovariate(service_rate) for _ in range(total_buses))
        visits, _, _ = simulate_corridor(
            ("One berth",), (), arrivals, (), one_stop_service_times=service
        )
        estimates.append(fmean(visits[bus, 0]["berth_queue_wait"]
                               for bus in range(warmup_buses + 1, total_buses + 1)))
    return expected, fmean(estimates), replications * measured_buses



def record_benchmark(label, result, tolerance):
    expected, actual, observations = result
    assert abs(actual - expected) <= tolerance, (label, expected, actual, tolerance)
    print(f"PASS: {label}: expected {expected:.2f} s, simulated {actual:.2f} s "
          f"(tolerance ±{tolerance:.1f} s, {observations:,} observations).")


record_benchmark("M/M/1 berth queue", verify_mm1(), 4.8)

In [ ]:
def verify_fixed_headway(replications=8, intervals=300, seed_start=3100):
    headway = 80
    end_seconds = headway * intervals
    # A bus boards at every boundary, including the end of the last measured interval.
    dispatches = tuple(range(0, end_seconds + 1, headway))
    estimates = []
    measured_passengers = 0
    for rep in range(replications):
        trips = generate_passenger_trips((6,), end_seconds, seed=seed_start + rep)
        visits, riders, _ = simulate_corridor(
            ("Origin", "Terminal"), (10,), dispatches, trips, dwell=(1, 0, 0)
        )
        assert all(visits[bus, 0]["service_start"] == dispatch
                   for bus, dispatch in enumerate(dispatches, start=1))
        estimates.append(fmean(rider["board"] - rider["arrival"] for rider in riders))
        measured_passengers += len(riders)
    return headway / 2, fmean(estimates), measured_passengers


record_benchmark("Fixed headway passenger wait", verify_fixed_headway(), 2.0)

In [ ]:
SCALING_REPS = (1, 2, 4, 8, 16, 32)
SCALING_BATCHES = 64
SCALING_SEED = 90_000
benchmark_wait = 40.0
batch_means = []
for batch in range(SCALING_BATCHES):
    start_seed = SCALING_SEED + batch * SCALING_REPS[-1]
    batch_means.append([
        verify_fixed_headway(replications=1, intervals=40, seed_start=start_seed + offset)[1]
        for offset in range(SCALING_REPS[-1])
    ])
scaling_rows = []
for count_reps in SCALING_REPS:
    rmse = sqrt(fmean((fmean(batch[:count_reps]) - benchmark_wait) ** 2 for batch in batch_means))
    scaling_rows.append({"replications": count_reps, "batches": SCALING_BATCHES, "rmse_seconds": rmse})
log_reps = [log(row["replications"]) for row in scaling_rows]
log_errors = [log(row["rmse_seconds"]) for row in scaling_rows]
mean_x, mean_y = fmean(log_reps), fmean(log_errors)
scaling_slope = (sum((x - mean_x) * (y - mean_y) for x, y in zip(log_reps, log_errors))
                 / sum((x - mean_x) ** 2 for x in log_reps))
assert scaling_rows[-1]["rmse_seconds"] < scaling_rows[0]["rmse_seconds"]
print(f"Monte Carlo RMSE slope: {scaling_slope:.3f}; theoretical reference: -0.500.")

fig, axis = plt.subplots(figsize=(7, 4.2), layout="constrained")
initial_error = scaling_rows[0]["rmse_seconds"]
axis.loglog(SCALING_REPS, [row["rmse_seconds"] for row in scaling_rows],
            "o-", color="#176789", label=f"Measured RMSE (slope {scaling_slope:.2f})")
axis.loglog(SCALING_REPS, [initial_error / sqrt(n) for n in SCALING_REPS],
            "--", color="#a44928", label="Reference slope −1/2")
axis.set_xticks(SCALING_REPS, labels=[str(n) for n in SCALING_REPS])
axis.set(title="Monte Carlo error · fixed-headway benchmark",
         xlabel="Independent replications in each estimate", ylabel="RMSE (seconds)")
axis.legend()
axis.grid(alpha=0.3, which="both")
fig.savefig("figures/monte-carlo-error-scaling.png", dpi=160, bbox_inches="tight")

## Default experiment and browser reference

The browser uses these exact seeded input manifests and road draws. Its controls vary the delay, demand or cap, and the defaults restore this reference.

In [ ]:
STOPS = ("Origin", "Market", "School", "Terminal")
MEAN_ROAD_SECONDS = (104, 98, 85)
BASE_RATES = tuple(0.35 * 60 / interval for interval in (13, 8, 10))
DISPATCHES = (60, 420, 780)
DEMANDS = {"Low": 0.60, "Medium": 1.00, "High": 1.75}
CAP, DELAY, TARGET_GAP = 20, 240, 360


def make_cohort(demand, seed):
    return generate_passenger_trips(tuple(demand * rate for rate in BASE_RATES), DISPATCHES[-1], seed)


def run_pair(cohort, draws, delay=DELAY, cap=CAP):
    options = {"running_time_draws": draws, "disruption": (1, 0, delay)}
    shared = (STOPS, MEAN_ROAD_SECONDS, DISPATCHES, cohort)
    baseline = simulate_corridor(*shared, **options)
    holding = simulate_corridor(*shared, **options, **(
        {"control_stop": 1, "target_gap": TARGET_GAP, "hold_cap": cap, "hold_buses": (2,)} if cap else {}
    ))
    for visits, riders, _ in (baseline, holding):
        assert [(r["arrival"], r["origin"], r["destination"]) for r in riders] == cohort
        assert all(r["alight"] is not None for r in riders)
        assert all(visits[bus, 1].get("hold", 0) == 0 for bus in (1, 3))
    return baseline, holding


def measures(run):
    visits, riders, _ = run
    school_boarding = [visits[bus, 2]["service_start"] for bus in (1, 2, 3)]
    for rider in riders:
        components = passenger_time_breakdown(visits, rider)
        assert isclose(sum(components.values()), rider["alight"] - rider["arrival"], abs_tol=1e-8)
    return {
        "completed": len(riders),
        "platform_wait": fmean(r["board"] - r["arrival"] for r in riders),
        "journey_time": fmean(r["alight"] - r["arrival"] for r in riders),
        "school_boarding": school_boarding,
        "school_gap_12": school_boarding[1] - school_boarding[0],
        "school_gap_23": school_boarding[2] - school_boarding[1],
        "bus2_hold": visits[2, 1].get("hold", 0),
        "market_berth_wait": fmean(visits[b, 1]["berth_queue_wait"] for b in (1, 2, 3)),
    }


example_trips = {label: make_cohort(demand, 42) for label, demand in DEMANDS.items()}
example_draws = sample_link_running_times(MEAN_ROAD_SECONDS, 0.15, 3, 73)
example = run_pair(example_trips["Medium"], example_draws)
zero = run_pair(example_trips["Medium"], example_draws, cap=0)
assert zero[0] == zero[1]
print("PASS: zero hold cap reproduces the full baseline records and trace.")
example_metrics = dict(zip(("baseline", "holding"), map(measures, example)))
for policy, result in example_metrics.items():
    print(policy, result)

## Matched experiments and confidence intervals

For each demand level and road-time CV, run 30 independent pairs. Draw passengers and road times once per pair. Report the holding-minus-baseline mean difference and its paired 95% Student t interval. Negative passenger differences favour holding. A shorter School gap can indicate bunching, so compare both gaps rather than interpreting one in isolation.

In [ ]:
from statistics import stdev
from csv import DictWriter
from json import dumps

POLICIES = ("baseline", "holding")
replication_rows = []
effect_rows = []
t_critical = 2.045229642  # Two-sided 95%, 29 degrees of freedom.
metrics = ("platform_wait", "journey_time", "school_gap_12", "school_gap_23", "market_berth_wait")
for demand_index, (demand, multiplier) in enumerate(DEMANDS.items()):
    for cv_index, cv in enumerate((0.15, 0.45)):
        scenario = 2 * demand_index + cv_index
        pairs = []
        for rep in range(30):
            demand_seed = 20_000 + 100 * scenario + rep
            road_seed = 40_000 + 100 * scenario + rep
            cohort = make_cohort(multiplier, demand_seed)
            draws = sample_link_running_times(MEAN_ROAD_SECONDS, cv, 3, road_seed)
            pair = [measures(run) for run in run_pair(cohort, draws)]
            pairs.append(pair)
            for policy, measured in zip(POLICIES, pair):
                replication_rows.append({
                    "demand": demand, "road_cv": cv, "replication": rep + 1,
                    "demand_seed": demand_seed, "road_seed": road_seed, "policy": policy,
                    **{k: v for k, v in measured.items() if k != "school_boarding"},
                })
        for metric in metrics:
            differences = [holding[metric] - baseline[metric] for baseline, holding in pairs]
            mean = fmean(differences)
            margin = t_critical * stdev(differences) / sqrt(len(differences))
            effect_rows.append({
                "demand": demand, "road_cv": cv, "metric": metric, "pairs": len(pairs),
                "difference": mean, "ci_low": mean - margin, "ci_high": mean + margin,
            })
            if metric in ("platform_wait", "journey_time"):
                print(f"{demand} CV {cv:.2f} {metric:14} {mean:+.2f} s [{mean-margin:+.2f}, {mean+margin:+.2f}]")


def save_csv(name, rows):
    with (Path("data") / name).open("w", newline="", encoding="utf-8") as file:
        writer = DictWriter(file, fieldnames=list(rows[0]), lineterminator="\n")
        writer.writeheader()
        writer.writerows(rows)


Path("data").mkdir(exist_ok=True)
save_csv("paired-replications.csv", replication_rows)
save_csv("policy-effects.csv", effect_rows)
save_csv("monte-carlo-scaling.csv", scaling_rows)
reference = {
    "tolerance_seconds": 1e-7,
    "defaults": {"demand": "Medium", "delay": DELAY, "cap": CAP},
    "stops": STOPS,
    "dispatches": DISPATCHES,
    "targetGap": TARGET_GAP,
    "runningTimes": example_draws,
    "trips": example_trips,
    "policies": example_metrics,
}
Path("data/browser-reference.json").write_text(dumps(reference, indent=2) + "\n", encoding="utf-8")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4), sharey=True, layout="constrained")
for axis, metric, label in zip(axes, metrics, ("Platform wait", "Journey time")):
    rows = [row for row in effect_rows if row["metric"] == metric]
    effects = [row["difference"] for row in rows]
    axis.errorbar(effects, range(len(rows)),
                  xerr=[[r["difference"] - r["ci_low"] for r in rows],
                        [r["ci_high"] - r["difference"] for r in rows]],
                  fmt="o", capsize=3)
    axis.axvline(0, color="#526671", lw=1)
    axis.set(title=label, xlabel="Holding minus baseline (s/passenger)")
    axis.set_yticks(range(len(rows)), [f"{r['demand']} · CV {r['road_cv']:.2f}" for r in rows])
axes[0].invert_yaxis()
fig.savefig("figures/paired-policy-effects.png", dpi=150, bbox_inches="tight")

## Discussion and limitations

In the default run, holding Bus 2 for 20 seconds widens B1–B2 at School by 24.27 seconds and reduces B2–B3 by 28.54 seconds. Across the same 89 passengers, mean platform wait falls 5.07 seconds and mean journey time falls 1.60 seconds. In the repeated experiment, four of six paired journey-time intervals favour holding. The low- and medium-demand intervals at higher road-time variability cross zero. These estimates describe a four-stop scenario with three buses, fixed dispatches, passenger-dependent dwell, assumed road-time variability, a fixed disruption and unlimited passenger capacity.

## 7. References

- [Herath et al. (2024), dwell equation 9 and table 3](https://www.jstage.jst.go.jp/article/easts/15/0/15_2109/_pdf/-char/en)
- [Jayasinghe et al. (2023), Galle Road bus operations](https://doi.org/10.4038/jsalt.v3i1.57)
- [Shen et al. (2026), headway irregularity and stop queues](https://doi.org/10.1016/j.tra.2025.104752)